# APK 6725 - AI for Sport & Movement Sciences
## Assignment 8: Trees, Forests and Honest Evaluation

This assignment practices the ideas from lectures 04.02 to 04.04. You will classify **daily activities** from smartphone sensor data.

- **Part A (2.5 points):** decision trees, random forests, and a comparison with logistic regression and kNN.
- **Part B (2.5 points):** how honest is the test? Data leakage, and how many subjects you need.

Write your code in the cells provided and test it. Some cells train many models: they can take 1 to 2 minutes in Colab.

Total: 5 points.

## Run this first

Run the cell below, at the start of every session. It downloads the course files to your Google Drive and keeps them up to date. When you see "Setup complete", you are ready to work.

**Do not edit or save your work inside the `AIforMovementScience` folder.** Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/Assignments'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
DATA = REPO + '/lectures/data'        # the folder that holds the data files
print('Setup complete. You can now run the rest of the notebook.')

## How to submit

Before you submit, please make sure your notebook is organized and reproducible. Part of your grade is for this.

- Answer each question in its own section, using the headings already provided.
- Put your code in the code cells; when a question asks for a written answer, use the markdown answer cell.
- Delete any scratch or experimental cells that are not part of your answer.
- Download your notebook: **File > Download > Download .ipynb**, name it `YourName_Assignment_8.ipynb`, and upload it to Canvas.

---
# The data: Human Activity Recognition (UCI HAR)

Thirty volunteers (19 to 48 years old) did six activities while they wore a smartphone on the waist:

1. Walking
2. Walking upstairs
3. Walking downstairs
4. Sitting
5. Standing
6. Laying

The phone recorded the accelerometer and the gyroscope at 50 samples per second. The authors cut the signals into **windows** of 2.56 s (128 samples). Consecutive windows **overlap by 50%**: each window shares half of its samples with the next window.

For each window, the authors computed **561 features**: for example the mean, the standard deviation, the maximum and the energy of each signal, in the time and in the frequency domain, and angles between the phone and gravity. So each row of the data is one window, with 561 features and one activity label.

The authors split the data **by subject**: 21 subjects in the training set and 9 subjects in the test set.

More information: <https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones>

The data is in `lectures/data/UCI_HAR_Dataset`. Run the cell below to load it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HAR = f"{DATA}/UCI_HAR_Dataset"
activities = ['Walking', 'Upstairs', 'Downstairs', 'Sitting', 'Standing', 'Laying']

# some of the 561 feature names are repeated, so we add a number to make each name unique
names = pd.read_csv(f"{HAR}/features.txt", sep=r"\s+", header=None)[1]
feature_names = [f"{n} [{i}]" for i, n in enumerate(names, start=1)]

def load(part):
    X = pd.read_csv(f"{HAR}/{part}/X_{part}.txt", sep=r"\s+", header=None, names=feature_names)
    y = pd.read_csv(f"{HAR}/{part}/y_{part}.txt", header=None)[0].map(lambda k: activities[k - 1])
    subjects = pd.read_csv(f"{HAR}/{part}/subject_{part}.txt", header=None)[0]
    return X, y, subjects

X_train, y_train, subjects_train = load('train')
X_test, y_test, subjects_test = load('test')
print('Data loaded.')
X_train.iloc[:5, :6]

---
# Part A - Trees and forests (2.5 points)

## Question A1 (0.25 points) - Look at the data

1. Print the shape of `X_train` and of `X_test`.
2. Print the number of different subjects in the train set and in the test set. Check that no subject is in both sets (hint: `set(subjects_train) & set(subjects_test)`).
3. Print the number of windows of each activity in the train set (`y_train.value_counts()`).
4. A very simple "model" always predicts the most common activity of the train set. Compute its accuracy on the **test** set.

In the answer cell: how many features does each window have? About how many windows does one subject give? What accuracy must a useful model beat?

In [ ]:
# 1. shapes
print('X_train:', X_train.shape)
print('X_test: ', X_test.shape)

# 2. subjects in each set, and check that no subject is in both
print('\nsubjects in the train set:', subjects_train.nunique())
print('subjects in the test set: ', subjects_test.nunique())
print('subjects in both sets:', set(subjects_train) & set(subjects_test))
print(f'windows per subject (train): {len(X_train) / subjects_train.nunique():.0f} on average '
      f'({subjects_train.value_counts().min()} to {subjects_train.value_counts().max()})')

# 3. windows of each activity in the train set
print('\nwindows per activity (train):')
print(y_train.value_counts())

# 4. always predict the most common activity of the train set
most_common = y_train.value_counts().idxmax()
baseline_accuracy = (y_test == most_common).mean()
print(f'\nmost common activity in the train set: {most_common}')
print(f'accuracy of always predicting {most_common} on the test set: {baseline_accuracy:.3f}')

**Your answer:** Each window has **561 features**. The train set has 7352 windows from 21 subjects and the test set 2947 windows from 9 subjects, and no subject is in both sets. One subject gives about **350 windows** (between 281 and 409 in the train set), about 50–60 per activity.

The most common activity in the train set is Laying (1407 windows). A model that always predicts Laying is right for **18.2%** of the test windows. A useful model must beat **0.182**. With six activities of similar size, guessing gives about 1 in 6.

## Question A2 (0.75 points) - One decision tree

Answer parts (a) and (b), each in its own cell.

### Part (a) - Train and test accuracy for each depth

For each `max_depth` in `[1, 2, 3, 4, 6, 8, 10, 15, 20]`, train a `DecisionTreeClassifier(max_depth=..., random_state=0)` on the train set. Store the accuracy on the train set and on the test set. Plot the two accuracies against the depth, on the same axes, with a legend.

**Note:** here we look at the test accuracy only to **see** the overfitting, as in lecture 04.02. To **select** a hyperparameter, use cross-validation on the train set (Question A3).

In the answer cell: what happens to the train accuracy and to the test accuracy when the depth increases? After which depth does the test accuracy stop improving? What does the gap between the two curves tell you?

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

depths = [1, 2, 3, 4, 6, 8, 10, 15, 20]
train_acc, test_acc = [], []
for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0)
    tree.fit(X_train, y_train)
    train_acc.append(tree.score(X_train, y_train))
    test_acc.append(tree.score(X_test, y_test))

display(pd.DataFrame({'train accuracy': train_acc, 'test accuracy': test_acc},
                     index=pd.Index(depths, name='max_depth')).round(3))

plt.figure(figsize=(7, 5))
plt.plot(depths, train_acc, 'o-', label='Train accuracy')
plt.plot(depths, test_acc, 's--', label='Test accuracy')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Decision tree: accuracy vs depth')
plt.xticks(depths)
plt.legend()
plt.grid()
plt.show()

**Your answer:** As the depth increases, the **train accuracy** keeps rising, from 0.38 at depth 1 to 0.99 at depth 10 and **1.00** at depth 20, where the tree memorizes every training window. The **test accuracy** rises quickly at first (0.36 → 0.81 at depth 4 → 0.86 at depth 6). It **stops improving after depth 8**, where it peaks at 0.871, and then stays flat at about 0.86.

The gap between the two curves grows with depth, from almost nothing at depths 1–2 to about 0.14 at depth 20 (1.00 vs 0.86). This is **overfitting**. The deep trees learn details of the 21 training subjects that do not carry over to the 9 new subjects. The extra depth only improves the fit to the training data, not to new people.

### Part (b) - Read a small tree

Train a tree with `max_depth=2` and plot it:

```Python
plt.figure(figsize=(16, 6))
plot_tree(tree_2, feature_names=feature_names, class_names=tree_2.classes_, filled=True, fontsize=9)
plt.show()
```

In the answer cell: which feature is used at the first split (the top of the tree)? Which activities go to the left and which go to the right? Explain why this split makes sense for the body. (Hint: look at the `value` row of each box. The order of the classes is `tree_2.classes_`.)

In [ ]:
tree_2 = DecisionTreeClassifier(max_depth=2, random_state=0)
tree_2.fit(X_train, y_train)
print('class order:', list(tree_2.classes_))

plt.figure(figsize=(16, 6))
plot_tree(tree_2, feature_names=feature_names, class_names=tree_2.classes_, filled=True, fontsize=9)
plt.show()

**Your answer:** The first split uses **`tGravityAcc-min()-X [53]`**, the minimum of the gravity component of the acceleration along the phone's X axis, with the rule ≤ 0.096.

- **Left (True):** all 1407 **Laying** windows and nothing else. This box is pure (gini = 0).
- **Right (False):** every other activity: Downstairs, Sitting, Standing, Upstairs and Walking.

This split makes sense for the body. The gravity component only depends on how the phone is **oriented**. With the phone on the waist, the X axis points along the body (vertical) when the person is upright, so gravity lies mostly along X. When the person lies down, the body (and the phone) turns about 90°, gravity no longer points along X, and the X gravity value drops. Laying is the only activity where the trunk is horizontal, so a single orientation feature separates it perfectly.

The second split on the right uses **`fBodyAccJerk-bandsEnergy()-1,16 [390]`**, the energy of the body jerk (movement intensity). It separates the static activities (Sitting, Standing) from the moving ones (Walking, Upstairs, Downstairs).

## Question A3 (1 point) - A random forest, tuned the right way

Answer parts (a) and (b), each in its own cell.

### Part (a) - Tune with group-aware cross-validation

Use `GridSearchCV` to select the hyperparameters of a random forest. Use `GroupKFold`, with the subjects as groups, so that the windows of one subject are never in the training part and in the validation part of the same fold.

```Python
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold

param_grid = {'max_depth': [5, 10, None], 'min_samples_leaf': [1, 5]}
search = GridSearchCV(RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1),
                      param_grid, cv=GroupKFold(n_splits=3), scoring='accuracy')
search.fit(X_train, y_train, groups=subjects_train)    # this takes about 1 minute
```

Print the best hyperparameters and the best cross-validation accuracy.

In the answer cell: why do we use `GroupKFold` here, and not the normal `KFold`? Is the test set used in this step?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold

param_grid = {'max_depth': [5, 10, None], 'min_samples_leaf': [1, 5]}
search = GridSearchCV(RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1),
                      param_grid, cv=GroupKFold(n_splits=3), scoring='accuracy')
search.fit(X_train, y_train, groups=subjects_train)    # this takes about 1 minute

print('best hyperparameters:', search.best_params_)
print(f'best cross-validation accuracy: {search.best_score_:.3f}')

# all the combinations, for reference
cv_table = pd.DataFrame(search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'mean_test_score', 'std_test_score']]
cv_table.sort_values('mean_test_score', ascending=False).round(3)

**Your answer:** The best hyperparameters are **`max_depth = 10`, `min_samples_leaf = 5`**, with a cross-validation accuracy of **0.894** (± 0.025). All six combinations are close, between 0.886 and 0.894.

We use **`GroupKFold`** because each subject gives about 350 windows, and windows of the same person are very similar: the same body, the same movement style, and windows that overlap by 50%. With the normal `KFold`, the windows of one subject would be spread over the training and validation parts of the same fold. The model would be validated on people it has already seen, and the validation accuracy would be too optimistic (data leakage). `GroupKFold` keeps every subject entirely in one part, so each fold is validated on **new people**, just like the final test.

**The test set is not used** in this step. `search.fit` only receives `X_train`, `y_train` and `subjects_train`, and the cross-validation splits only the 21 train subjects. The 9 test subjects stay untouched until part (b).

### Part (b) - Evaluate on the test set

Use the best model (`search.best_estimator_`). On the test set:

1. Print the accuracy.
2. Print the `classification_report`.
3. Plot the confusion matrix:
   ```Python
   from sklearn.metrics import classification_report, ConfusionMatrixDisplay
   ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=activities, xticks_rotation=45, cmap='Blues')
   ```

In the answer cell:
- Which two activities does the model confuse most? Why does this make sense for a phone on the waist?
- Look at the precision and the recall of `Sitting`. Say in words what each number means for this activity (lecture 04.02).

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

rf = search.best_estimator_
pred = rf.predict(X_test)

print(f'test accuracy: {rf.score(X_test, y_test):.3f}\n')
print(classification_report(y_test, pred, labels=activities, digits=3))

ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=activities, xticks_rotation=45, cmap='Blues')
plt.title('Random forest, test set')
plt.show()

**Your answer:** The test accuracy is **0.922**, a little higher than the cross-validation estimate of 0.894.

- **Most confused activities:** **Sitting and Standing**. 56 Sitting windows were predicted as Standing and 44 Standing windows as Sitting, 100 errors in total, the most of any pair. This makes sense for a phone on the waist: in both activities the person is still and the trunk is upright, so the phone sees almost the same thing, gravity pointing down and almost no movement. The difference is in the legs (thighs horizontal or vertical), which a sensor on the waist cannot see. The next most confused group is the three walking activities (Walking, Upstairs, Downstairs), which are also similar movements. Laying is perfect (precision and recall = 1.00), because of the phone's orientation (part A2b).
- **Precision of Sitting = 0.908:** when the model says "Sitting", it is right 90.8% of the time. The other 9.2% are mostly Standing windows that were labelled Sitting (false alarms for Sitting).
- **Recall of Sitting = 0.886:** of all the windows where the person really was sitting, the model found 88.6%. The other 11.4% (56 windows) were missed, all labelled Standing.

## Question A4 (0.5 points) - Compare with other models, and look inside the forest

Answer parts (a) and (b), each in its own cell.

### Part (a) - Logistic regression and kNN

Train these two models on the train set and print their test accuracy. Both use distances or coefficients, so they need scaled features.

```Python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

lr  = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000))])
knn = Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier(n_neighbors=15))])
```

In the answer cell: rank the three models (random forest, logistic regression, kNN) by test accuracy. Is the most flexible model always the best? Why can a linear model work well with these 561 features?

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

lr  = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000))])
knn = Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier(n_neighbors=15))])

lr.fit(X_train, y_train)
knn.fit(X_train, y_train)

model_comparison = pd.Series({
    'Random forest':       rf.score(X_test, y_test),
    'Logistic regression': lr.score(X_test, y_test),
    'kNN (k = 15)':        knn.score(X_test, y_test),
}, name='test accuracy').sort_values(ascending=False)
model_comparison.round(3)

**Your answer:** The ranking by test accuracy is:

1. **Logistic regression: 0.955**
2. **Random forest: 0.922**
3. **kNN (k = 15): 0.893**

The most flexible model is **not** the best. The random forest can draw very complex boundaries, but the simple linear model beats it by about 3 percentage points.

A linear model can work well here because the 561 features were already **engineered** by the authors: means, standard deviations, energies, frequency bands and angles with gravity. These features turn the raw signals into quantities that separate the activities almost in straight lines. For example, one gravity feature alone separates Laying perfectly (A2b). With so many informative features, the classes are close to **linearly separable**, so a straight boundary is enough. A linear model with many inputs is also less likely to fit the movement style of the 21 training subjects (it is regularized), so it generalizes well to new people. kNN suffers from the many dimensions: with 561 features, the distances between windows become less informative, and many features are noisy or redundant.

### Part (b) - Feature importance

Make a horizontal bar plot of the 10 most important features of the random forest (`rf.feature_importances_`). Use the names in `feature_names`.

In the answer cell: what type of information do most of the top features measure? (Hint: in the feature names, `Gravity` is the slow part of the acceleration, caused by gravity; `Body` is the fast part, caused by the movement; `angle(...)` is an angle between the phone and gravity.) Why are these features useful to separate the activities?

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
top10 = importances.head(10)
print(top10.round(4))

plt.figure(figsize=(9, 5))
plt.barh(top10.index[::-1], top10.values[::-1])     # reversed so the most important is at the top
plt.xlabel('Feature importance (mean decrease in impurity)')
plt.title('Random forest: 10 most important features')
plt.tight_layout()
plt.show()

**Your answer:** 9 of the 10 most important features measure **gravity, that is, the orientation of the phone** (and so of the trunk):

- `angle(X,gravityMean)` and `angle(Y,gravityMean)`: the angle between the phone's axes and gravity.
- `tGravityAcc-min/mean/max/energy` along X and Y: the slow, gravity part of the acceleration.

The only exception is `fBodyAccJerk-bandsEnergy()-1,8`, the energy of the body jerk at low frequencies, which measures **how much the body moves**.

These features are useful because they measure the two things that separate the six activities:

1. **Posture:** the orientation of the trunk relative to gravity. Laying has a completely different orientation from the upright activities. Sitting and standing differ slightly in how the trunk and pelvis are tilted, and walking adds small changes in tilt.
2. **Movement:** the jerk energy separates the still activities (Sitting, Standing, Laying) from the moving ones (Walking, Upstairs, Downstairs).

Gravity features are also very stable from one person to another, because gravity is the same for everyone, which makes them reliable for new subjects.

---
# Part B - How honest is the test? (2.5 points)

In Part A, the authors split the data by subject. What happens if we split the windows at random? And how many subjects do we need for a good model?

## Question B1 (1 point) - Data leakage: split by window or by subject?

Put all 30 subjects together:

```Python
X_all = pd.concat([X_train, X_test], ignore_index=True)
y_all = pd.concat([y_train, y_test], ignore_index=True)
subjects_all = pd.concat([subjects_train, subjects_test], ignore_index=True)
```

Repeat this experiment for `seed` in `[0, 1, 2]`:

1. **Split by window:** `train_test_split(X_all, y_all, test_size=0.3, random_state=seed, stratify=y_all)`.
2. **Split by subject:** `GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)` with `groups=subjects_all`. (Lecture 04.03 shows how to get `train_idx` and `test_idx`.)
3. For each split, train `RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)` on the train part and compute the accuracy on the test part.

Print the six accuracies, and the mean accuracy of each type of split.

In the answer cell:
- Which split gives the higher accuracy?
- Give **two** reasons why. (Hint: remember how the windows were made, and think about the movement style of each person.)
- A clinic wants to use the model on **new** patients. Which number should you report to the clinic?
- In lecture 04.04, the split by row and the split by subject gave almost the same AUC for the PD data. Why is the difference much larger here?

In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit

X_all = pd.concat([X_train, X_test], ignore_index=True)
y_all = pd.concat([y_train, y_test], ignore_index=True)
subjects_all = pd.concat([subjects_train, subjects_test], ignore_index=True)

leak_results = []
for seed in [0, 1, 2]:
    # 1. split by window (random rows)
    Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.3, random_state=seed, stratify=y_all)
    model = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
    model.fit(Xtr, ytr)
    acc_window = model.score(Xte, yte)

    # 2. split by subject
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)
    train_idx, test_idx = next(gss.split(X_all, y_all, groups=subjects_all))
    model = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
    model.fit(X_all.iloc[train_idx], y_all.iloc[train_idx])
    acc_subject = model.score(X_all.iloc[test_idx], y_all.iloc[test_idx])

    leak_results.append({'seed': seed, 'split by window': acc_window, 'split by subject': acc_subject})
    print(f'seed {seed}: split by window = {acc_window:.3f}, split by subject = {acc_subject:.3f} '
          f'({subjects_all.iloc[test_idx].nunique()} test subjects)')

leak_results = pd.DataFrame(leak_results).set_index('seed')
print('\nmean accuracy:')
print(leak_results.mean().round(3))

**Your answer:**

| Seed | Split by window | Split by subject |
| --- | --- | --- |
| 0 | 0.973 | 0.908 |
| 1 | 0.978 | 0.922 |
| 2 | 0.972 | 0.963 |
| **Mean** | **0.975** | **0.931** |

- **The split by window** gives the higher accuracy: about 4–5 percentage points higher on average, and higher for every seed. Its results are also very stable (0.972–0.978), while the split by subject changes more (0.908–0.963) depending on which 9 people end up in the test set.
- **Two reasons:**
  1. **Overlapping windows.** Each window shares 50% of its samples with the next window. With a random split, a test window almost always has its neighbours in the train set, and they contain half of the same signal. The model is tested on near-copies of windows it has already seen.
  2. **Each person's own movement style.** Every subject walks, sits and climbs stairs in their own way: body size, gait pattern, how the phone sits on the waist. With a split by window, all 30 people are in the train set, so the model can recognize the person's style instead of learning what the activity looks like in general. With a split by subject, the test people are truly new.
- **The number to report to the clinic** is the **split by subject**, about **0.93** (with its spread, 0.91–0.96 here). New patients are, by definition, people the model has never seen, and that is exactly what the split by subject simulates. The 0.975 from the split by window is too optimistic for new patients.
- **Why the difference is much larger here than in lecture 04.04:** in the PD data, most subjects had only one or two rows, and their recordings were not identical, so a split by row barely leaked any information. Here each subject gives about **350 windows**, all very similar and overlapping by 50%. So a random split puts hundreds of near-duplicate rows of the same person on both sides, and the leakage is much stronger.

## Question B2 (1 point) - How many subjects do you need?

Keep the official test set (9 subjects). Train the random forest with only **some** of the 21 train subjects, and see how the test accuracy changes. For each number of subjects, repeat 5 times with different subjects chosen at random:

```Python
rng = np.random.RandomState(0)
train_ids = subjects_train.unique()
n_list = [2, 4, 8, 14, 21]
mean_acc, std_acc = [], []
for n in n_list:
    accs = []
    for repeat in range(5):
        chosen = rng.choice(train_ids, n, replace=False)
        rows = subjects_train.isin(chosen)
        model = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
        model.fit(X_train[rows], y_train[rows])
        accs.append(model.score(X_test, y_test))
    # complete the loop: store the mean and the standard deviation of accs
```

This takes about 1 minute. Then plot the mean test accuracy against the number of subjects, with the standard deviation as error bars (`plt.errorbar(n_list, mean_acc, yerr=std_acc, marker='o', capsize=4)`). Label the axes.

In the answer cell:
- How do the mean and the spread of the test accuracy change with the number of subjects?
- Why is the standard deviation zero for 21 subjects?
- Two subjects give about 700 windows. Is that a lot of data? Explain the difference between **many samples** and **many subjects**.
- Would a model trained on 4 subjects be a good proof of principle? Would it be a good final model?

In [ ]:
rng = np.random.RandomState(0)
train_ids = subjects_train.unique()
n_list = [2, 4, 8, 14, 21]
mean_acc, std_acc = [], []
for n in n_list:
    accs = []
    for repeat in range(5):
        chosen = rng.choice(train_ids, n, replace=False)
        rows = subjects_train.isin(chosen)
        model = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
        model.fit(X_train[rows], y_train[rows])
        accs.append(model.score(X_test, y_test))
    # complete the loop: store the mean and the standard deviation of accs
    mean_acc.append(np.mean(accs))
    std_acc.append(np.std(accs))
    print(f'{n:2d} subjects ({rows.sum()} windows in the last repeat): '
          f'mean = {np.mean(accs):.3f}, std = {np.std(accs):.3f}, min = {np.min(accs):.3f}, max = {np.max(accs):.3f}')

plt.figure(figsize=(7, 5))
plt.errorbar(n_list, mean_acc, yerr=std_acc, marker='o', capsize=4)
plt.xlabel('Number of training subjects')
plt.ylabel('Test accuracy (9 official test subjects)')
plt.title('Random forest: test accuracy vs number of training subjects')
plt.xticks(n_list)
plt.grid()
plt.show()

**Your answer:**

| Training subjects | Mean test accuracy | SD |
| --- | --- | --- |
| 2 | 0.778 | 0.009 |
| 4 | 0.847 | 0.026 |
| 8 | 0.887 | 0.013 |
| 14 | 0.907 | 0.026 |
| 21 | 0.926 | 0.000 |

- **Mean and spread:** the mean test accuracy rises steadily with the number of subjects, from 0.78 with 2 subjects to 0.93 with 21. The gain is large at first (+0.07 from 2 to 4) and smaller later (+0.02 from 14 to 21), so the curve starts to flatten. The spread does not follow a clean trend with only 5 repeats: it is between about 0.01 and 0.03 for 2 to 14 subjects. The worst result of a single draw generally gets better as subjects are added (0.76 → 0.81 → 0.87 → 0.86 → 0.93), and which subjects you happen to pick matters (for example, 0.86 to 0.94 with 14 subjects).
- **Why the SD is zero for 21 subjects:** there are only 21 train subjects, so choosing 21 of them without replacement always gives **all** of them. The 5 repeats train on exactly the same data with the same `random_state=0`, so they build the same forest and get the same accuracy every time.
- **About 700 windows from 2 subjects:** that is **many samples** but very **little variety**. All 700 windows come from 2 people, so they show only 2 bodies, 2 movement styles and 2 ways of wearing the phone, and the overlapping windows are partly copies of each other. The model does not learn how *people* in general perform the activities, only how these two people do. **Many subjects** means many different people, and that is what lets a model generalize to a new person. For a model meant for new people, the number of subjects matters more than the number of rows.
- **A model trained on 4 subjects:** it is a reasonable **proof of principle**. It already reaches about 0.85, far above the 0.18 baseline, which shows that the sensor and features carry the information needed. It would **not** be a good **final model**: it is about 8 points worse than with 21 subjects, it depends on which 4 people were chosen, and 4 people cannot represent the variety of the people a final model would be used on.

## Question B3 (0.5 points) - Read a result critically

A research paper reports a model that detects freezing of gait in Parkinson's disease with **99% accuracy**. The authors recorded 5 patients with a sensor on the ankle, cut the signals into 2-second windows with 50% overlap, and split the **windows** at random into 80% train and 20% test.

In the answer cell:
- Give **two** problems with this evaluation. Use what you learned in B1, B2 and lecture 04.04.
- Describe how you would evaluate the model in a more honest way.
- Accuracy may not be the best metric here: freezing happens in only a small part of the time. Which metrics from lecture 04.02 would you report, and why?

**Your answer:**

**Two problems with this evaluation**

1. **Data leakage from the random split of windows (B1).** The windows overlap by 50%, so neighbouring windows share half their signal, and a random 80/20 split puts near-copies of the test windows in the train set. The same 5 patients are also in both sets, so the model can learn each patient's own way of freezing and walking instead of freezing in general. The 99% measures how well the model recognizes windows of patients it has already seen, not new patients.
2. **Too few patients (B2, lecture 04.04).** 5 patients cannot represent the variety of people with PD: freezing looks different from person to person and with disease stage and medication. With so few subjects, the result depends heavily on who was recorded, and a different 5 patients could give a very different number. At best it is a proof of principle.

**A more honest evaluation**
- Split **by patient**. With few patients, use **leave-one-subject-out** cross-validation (`GroupKFold` with the patients as groups): train on 4 patients, test on the 5th, and repeat for each patient.
- Do all model selection and tuning inside the training patients only (group-aware cross-validation), and keep a test set of patients that is used once.
- Report the **mean and the spread** over the test patients, not one number, and record **more patients** before claiming a usable model.

**Better metrics than accuracy**

Freezing happens in only a small part of the time, so a model that always says "no freezing" already has a very high accuracy (for example, 90% if freezing is 10% of the windows) while detecting nothing. I would report:

- the **confusion matrix**, so we see both types of error
- the **recall (sensitivity)** for freezing: what fraction of the real freezing episodes the model detects. Missed freezes matter for patient safety.
- the **precision** for freezing: when the model says "freezing", how often it is right. Too many false alarms make a device useless.
- the **F1 score** or **balanced accuracy**, which combine both and are not inflated by the large "no freezing" class
- the **ROC AUC**, which shows how well the model ranks freezing above non-freezing over all thresholds

---
## Extra (no points)

- In A4(a), try `n_neighbors` = 1, 5, 50 and 150 for kNN. Which value is best? Is the result the same as with the moons data in lecture 04.02?
- Train the random forest with only the features that have `angle` in their name (`[f for f in feature_names if 'angle' in f]`). How many features is this? What accuracy do you get?